In [1]:
# 高性能VAE缺失值填补模型 - 平衡优化版本
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers, optimizers, regularizers, initializers, callbacks
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, QuantileTransformer
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, VotingRegressor, AdaBoostRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge, HuberRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from scipy import stats
import warnings
import random
import os
import gc
import time

warnings.filterwarnings("ignore")

# 检查TensorFlow版本
print(f"TensorFlow版本: {tf.__version__}")

# ==================== 强化的可重复性设置 ====================
SEED = 42

def set_all_seeds(seed=42):
    """设置所有随机种子以确保可重复性"""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

# 初始化随机种子
set_all_seeds(SEED)

# GPU内存设置
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"发现 {len(gpus)} 个GPU设备")
    except RuntimeError as e:
        print(f"GPU设置错误: {e}")
else:
    print("未发现GPU，使用CPU运行")

# 清理会话
tf.keras.backend.clear_session()
gc.collect()

# 1. 数据加载和预处理
def load_and_preprocess_data():
    """加载数据并进行初步预处理"""
    data = pd.read_csv("./Earthquake.csv", low_memory=True)
    print(f"原始数据形状: {data.shape}")
    
    # 保存原始数据副本
    original_data = data.copy()
    
    # 删除非数值列
    cols_to_drop = []
    for col in data.columns:
        try:
            pd.to_numeric(data[col].dropna().iloc[:10])
        except:
            cols_to_drop.append(col)
    
    data = data.drop(columns=cols_to_drop)
    
    # 删除ID列
    for col in data.columns[:3]:
        if data[col].dtype in ['int64', 'float64']:
            non_null = data[col].dropna()
            if len(non_null) > 1:
                diffs = non_null.diff().dropna()
                if (diffs == 1).mean() > 0.9:
                    data = data.drop(columns=[col])
    
    # 删除L(m)列
    if 'L(m)' in data.columns:
        data = data.drop(columns=['L(m)'])
    
    # 移除缺失率过高的列（>50%）
    missing_ratio = data.isnull().sum() / len(data)
    cols_to_drop = missing_ratio[missing_ratio > 0.50].index
    data = data.drop(columns=cols_to_drop)
    
    # 转换数据类型
    for col in data.columns:
        data[col] = pd.to_numeric(data[col], errors='coerce').astype('float32')
    
    print(f"最终数据形状: {data.shape}")
    print(f"特征: {list(data.columns)}")
    
    return data, original_data

# 2. 平衡的异常值处理
def adaptive_outlier_handling(df):
    """平衡的异常值处理策略"""
    df_clean = df.copy()
    transform_info = {}
    
    for column in df_clean.columns:
        mask = df_clean[column].notna()
        values = df_clean.loc[mask, column].values
        
        if len(values) == 0:
            continue
        
        # 计算偏度和峰度
        skewness = stats.skew(values)
        kurtosis = stats.kurtosis(values)
        
        # 平衡的异常值处理
        if abs(skewness) > 2:  # 高度偏斜
            # 使用适中的分位数
            lower, upper = np.percentile(values, [0.25, 99.75])
        else:  # 接近正态
            # 使用标准IQR方法
            q1, q3 = np.percentile(values, [25, 75])
            iqr = q3 - q1
            # 使用标准系数2.75
            lower = q1 - 2.75 * iqr
            upper = q3 + 2.75 * iqr
            
            # 确保不丢失太多数据
            lower = max(lower, np.percentile(values, 0.5))
            upper = min(upper, np.percentile(values, 99.5))
        
        # 软裁剪
        values_clipped = np.clip(values, lower, upper)
        
        transform_info[column] = {
            'type': 'balanced_clip',
            'lower': lower,
            'upper': upper,
            'skewness': skewness,
            'kurtosis': kurtosis
        }
        
        df_clean.loc[mask, column] = values_clipped
    
    return df_clean, transform_info

# 3. 平衡的标准化策略
def optimized_normalization(data):
    """平衡的标准化策略"""
    data_normalized = pd.DataFrame(index=data.index, columns=data.columns)
    scalers = {}
    
    for column in data.columns:
        mask = data[column].notna()
        values = data[column][mask].values.reshape(-1, 1)
        
        if len(values) > 0:
            # 计算数据特征
            skewness = stats.skew(values.flatten())
            value_range = values.max() - values.min()
            
            # 平衡的标准化选择
            if abs(skewness) > 1.5 or value_range > 1000:
                # 使用分位数变换
                scaler = QuantileTransformer(
                    output_distribution='normal',  # 使用正态分布
                    n_quantiles=min(750, len(values)),  # 适中的分位数
                    subsample=100000,
                    random_state=42
                )
            elif column in ['H(m)', 'At(m²）', 'Wa(m)']:
                # 特定特征使用RobustScaler
                scaler = RobustScaler(quantile_range=(15, 85))  # 平衡的分位数范围
            else:
                # 默认使用StandardScaler
                scaler = StandardScaler()
            
            scaler.fit(values)
            scaled = scaler.transform(values).flatten()
            
            # 映射到[0.15, 0.85]（平衡范围）
            min_val, max_val = scaled.min(), scaled.max()
            if max_val - min_val > 1e-8:
                scaled = (scaled - min_val) / (max_val - min_val) * 0.7 + 0.15
            else:
                scaled = np.ones_like(scaled) * 0.5
            
            data_normalized.loc[mask, column] = scaled.astype(np.float32)
            scalers[column] = {
                'scaler': scaler,
                'min': min_val,
                'max': max_val,
                'type': type(scaler).__name__
            }
    
    return data_normalized, scalers

# 4. 改进的反标准化
def inverse_transform_optimized(data, scalers, column_names):
    """优化的反标准化"""
    data_original = np.zeros_like(data)
    
    for col_idx, col_name in enumerate(column_names):
        values = data[:, col_idx].copy()
        
        if col_name in scalers:
            scaler_info = scalers[col_name]
            
            # 反映射
            values = (values - 0.15) / 0.7
            values = values * (scaler_info['max'] - scaler_info['min']) + scaler_info['min']
            
            # 反标准化
            values = scaler_info['scaler'].inverse_transform(values.reshape(-1, 1)).flatten()
        
        data_original[:, col_idx] = values
    
    return data_original

# 5. 平衡的特征工程
def enhanced_feature_engineering(data, mask):
    """平衡的特征工程"""
    n_samples, n_features = data.shape
    
    # 基础特征
    features = [data]
    
    # 添加平方根特征
    sqrt_data = np.sqrt(np.abs(data) + 1e-8) * np.sign(data)
    features.append(sqrt_data)
    
    # 添加少量交互特征（仅对3个特征）
    if n_features <= 3:
        for i in range(n_features):
            for j in range(i+1, min(i+2, n_features)):
                interaction = data[:, i:i+1] * data[:, j:j+1]
                features.append(interaction)
    
    # 合并所有特征
    enhanced_data = np.hstack(features)
    
    # 扩展掩码
    n_enhanced = enhanced_data.shape[1]
    n_base = data.shape[1]
    enhanced_mask = np.tile(mask, (1, n_enhanced // n_base))
    if enhanced_mask.shape[1] < n_enhanced:
        enhanced_mask = np.hstack([enhanced_mask, mask[:, :n_enhanced - enhanced_mask.shape[1]]])
    
    return enhanced_data, enhanced_mask

# 6. 平衡的VAE模型
class OptimizedVAE:
    """平衡的VAE模型"""
    def __init__(self, data_dim, enhanced_dim, latent_dim=7, learning_rate=0.0006, seed=42):
        self.data_dim = data_dim
        self.enhanced_dim = enhanced_dim
        self.latent_dim = latent_dim  # 平衡的潜在维度
        self.learning_rate = learning_rate  # 平衡的学习率
        self.seed = seed
        
        tf.random.set_seed(seed)
        
        self.encoder = self._build_encoder()
        self.decoder = self._build_decoder()
        
        # 平衡的优化器设置
        self.optimizer = optimizers.Adam(
            learning_rate=learning_rate,
            beta_1=0.9,
            beta_2=0.999,
            epsilon=1e-7
        )
        
        self.best_weights = None
        self.best_val_loss = float('inf')
        self.training_history = {'train_loss': [], 'val_loss': []}
    
    def _build_encoder(self):
        """构建平衡的编码器"""
        inputs = layers.Input(shape=(self.enhanced_dim,))
        
        # 平衡的网络结构
        x = layers.Dense(20, activation='relu',
                        kernel_initializer=initializers.HeNormal(seed=self.seed),
                        kernel_regularizer=regularizers.L2(0.015))(inputs)  # 适中的正则化
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.35)(x)  # 平衡的dropout
        
        x = layers.Dense(14, activation='relu',
                        kernel_initializer=initializers.HeNormal(seed=self.seed),
                        kernel_regularizer=regularizers.L2(0.015))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.35)(x)
        
        # 潜在空间
        z_mean = layers.Dense(self.latent_dim,
                             kernel_regularizer=regularizers.L2(0.01))(x)
        z_log_var = layers.Dense(self.latent_dim,
                                kernel_regularizer=regularizers.L2(0.01))(x)
        
        # 平衡的方差限制
        z_log_var = layers.Lambda(lambda x: tf.clip_by_value(x, -2.5, 1.5))(z_log_var)
        
        return keras.Model(inputs, [z_mean, z_log_var])
    
    def _build_decoder(self):
        """构建平衡的解码器"""
        inputs = layers.Input(shape=(self.latent_dim,))
        
        x = layers.Dense(14, activation='relu',
                        kernel_initializer=initializers.HeNormal(seed=self.seed),
                        kernel_regularizer=regularizers.L2(0.015))(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.35)(x)
        
        x = layers.Dense(20, activation='relu',
                        kernel_initializer=initializers.HeNormal(seed=self.seed),
                        kernel_regularizer=regularizers.L2(0.015))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.35)(x)
        
        # 输出层
        outputs = layers.Dense(self.data_dim, activation='sigmoid',
                              kernel_regularizer=regularizers.L2(0.008))(x)
        
        return keras.Model(inputs, outputs)
    
    def _sample(self, z_mean, z_log_var):
        """重参数化技巧（平衡版）"""
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim), seed=self.seed)
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon * 0.12  # 平衡的噪声
    
    @tf.function
    def train_step(self, x_batch_enhanced, x_batch_original, mask_batch, beta):
        """训练步骤"""
        with tf.GradientTape() as tape:
            z_mean, z_log_var = self.encoder(x_batch_enhanced, training=True)
            z = self._sample(z_mean, z_log_var)
            reconstructed = self.decoder(z, training=True)
            
            # 混合损失：MSE和Huber的组合
            diff = x_batch_original - reconstructed
            
            # MSE部分（权重0.6）
            mse_loss = tf.square(diff) * mask_batch
            
            # Huber部分（权重0.4）
            huber_delta = 0.5
            is_small_error = tf.abs(diff) <= huber_delta
            small_error_loss = tf.square(diff) / 2
            large_error_loss = huber_delta * (tf.abs(diff) - huber_delta / 2)
            huber_loss = tf.where(is_small_error, small_error_loss, large_error_loss) * mask_batch
            
            # 组合重建损失
            reconstruction_loss = (0.6 * tf.reduce_sum(mse_loss) + 
                                 0.4 * tf.reduce_sum(huber_loss)) / tf.reduce_sum(mask_batch)
            
            # KL损失（平衡权重）
            kl_loss = -0.5 * tf.reduce_mean(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
            )
            
            # 总损失
            total_loss = reconstruction_loss + beta * kl_loss
            
            # L2正则化
            l2_loss = tf.add_n(self.encoder.losses + self.decoder.losses)
            total_loss += l2_loss
        
        variables = self.encoder.trainable_variables + self.decoder.trainable_variables
        gradients = tape.gradient(total_loss, variables)
        
        # 平衡的梯度裁剪
        gradients = [tf.clip_by_norm(g, 0.75) for g in gradients]
        self.optimizer.apply_gradients(zip(gradients, variables))
        
        return total_loss, reconstruction_loss, kl_loss
    
    def fit(self, data_enhanced, data_original, mask, validation_split=0.2,
            epochs=90, batch_size=96):  # 平衡的参数
        """训练模型"""
        n_samples = len(data_original)
        indices = np.arange(n_samples)
        np.random.seed(self.seed)
        np.random.shuffle(indices)
        
        split_point = int(n_samples * (1 - validation_split))
        train_indices = indices[:split_point]
        val_indices = indices[split_point:]
        
        x_train_enhanced = data_enhanced[train_indices]
        x_train_original = data_original[train_indices]
        x_val_enhanced = data_enhanced[val_indices]
        x_val_original = data_original[val_indices]
        mask_train = mask[train_indices]
        mask_val = mask[val_indices]
        
        print(f"训练集: {len(x_train_original)} 样本")
        print(f"验证集: {len(x_val_original)} 样本")
        
        patience = 18  # 平衡的patience
        patience_counter = 0
        min_improvement = 0.0003  # 平衡的改进阈值
        
        # KL退火
        kl_anneal_epochs = 25
        
        for epoch in range(epochs):
            # 平衡的KL退火
            beta = min(0.8, epoch / kl_anneal_epochs) * 0.008
            
            train_indices_shuffled = np.random.permutation(len(x_train_original))
            train_loss = 0
            train_recon_loss = 0
            train_kl_loss = 0
            n_batches = 0
            
            for i in range(0, len(x_train_original), batch_size):
                batch_indices = train_indices_shuffled[i:i+batch_size]
                loss, recon_loss, kl_loss = self.train_step(
                    x_train_enhanced[batch_indices],
                    x_train_original[batch_indices],
                    mask_train[batch_indices],
                    beta
                )
                train_loss += loss
                train_recon_loss += recon_loss
                train_kl_loss += kl_loss
                n_batches += 1
            
            train_loss /= n_batches
            train_recon_loss /= n_batches
            train_kl_loss /= n_batches
            self.training_history['train_loss'].append(float(train_loss))
            
            # 验证
            if epoch % 5 == 0:
                val_loss = 0
                n_val_batches = 0
                
                for i in range(0, len(x_val_original), batch_size):
                    z_mean, z_log_var = self.encoder(x_val_enhanced[i:i+batch_size], training=False)
                    z = self._sample(z_mean, z_log_var)
                    reconstructed = self.decoder(z, training=False)
                    
                    diff = x_val_original[i:i+batch_size] - reconstructed
                    mse = tf.square(diff) * mask_val[i:i+batch_size]
                    val_loss += tf.reduce_sum(mse) / tf.reduce_sum(mask_val[i:i+batch_size])
                    n_val_batches += 1
                
                val_loss /= n_val_batches
                self.training_history['val_loss'].append(float(val_loss))
                
                # 早停
                if val_loss < self.best_val_loss - min_improvement:
                    self.best_val_loss = val_loss
                    self.best_weights = {
                        'encoder': self.encoder.get_weights(),
                        'decoder': self.decoder.get_weights()
                    }
                    patience_counter = 0
                else:
                    patience_counter += 5
                
                if epoch % 20 == 0:
                    print(f"Epoch {epoch}: Train Loss={train_loss:.6f}, "
                          f"Recon={train_recon_loss:.6f}, KL={train_kl_loss:.6f}, "
                          f"Val Loss={val_loss:.6f}")
                
                if patience_counter >= patience:
                    print(f"早停于第 {epoch} 轮")
                    break
        
        # 恢复最佳权重
        if self.best_weights:
            self.encoder.set_weights(self.best_weights['encoder'])
            self.decoder.set_weights(self.best_weights['decoder'])
    
    def impute(self, data_enhanced, data_original, mask):
        """填补缺失值（平衡策略）"""
        imputed = np.zeros_like(data_original)
        batch_size = 192
        
        for i in range(0, len(data_original), batch_size):
            batch_enhanced = data_enhanced[i:i+batch_size]
            batch_original = data_original[i:i+batch_size]
            batch_mask = mask[i:i+batch_size]
            
            # 平衡的采样策略
            predictions = []
            for _ in range(4):  # 4次采样
                z_mean, z_log_var = self.encoder(batch_enhanced, training=False)
                # 混合策略：部分使用均值，部分使用采样
                if _ < 2:
                    z = z_mean  # 前两次使用均值
                else:
                    z = self._sample(z_mean, z_log_var)  # 后两次使用采样
                pred = self.decoder(z, training=False).numpy()
                predictions.append(pred)
            
            # 使用加权平均
            pred_mean = np.mean(predictions[:2], axis=0) * 0.6 + np.mean(predictions[2:], axis=0) * 0.4
            
            imputed[i:i+batch_size] = batch_original * batch_mask + pred_mean * (1 - batch_mask)
        
        return imputed

# 7. 平衡的目标特征模型
class EnhancedTargetedModel:
    """平衡的目标特征专用模型"""
    def __init__(self, target_features, seed=42):
        self.target_features = target_features
        self.seed = seed
        self.base_models = {}
        
    def _get_model_for_feature(self, col_name):
        """为特定特征返回平衡的模型配置"""
        if col_name == 'H(m)':
            # 平衡的集成模型
            models = [
                ('rf', RandomForestRegressor(
                    n_estimators=25,  # 平衡的树数量
                    max_depth=4,  # 平衡的深度
                    min_samples_split=25,
                    min_samples_leaf=12,
                    max_features=0.4,  # 平衡的特征比例
                    bootstrap=True,
                    random_state=self.seed
                )),
                ('ridge', Ridge(alpha=2.0, random_state=self.seed)),  # 平衡的正则化
                ('gbr', GradientBoostingRegressor(
                    n_estimators=20,
                    max_depth=3,
                    learning_rate=0.04,
                    random_state=self.seed
                ))
            ]
            return VotingRegressor(models)
            
        elif col_name == 'At(m²）':
            # 平衡的RandomForest
            return RandomForestRegressor(
                n_estimators=30,
                max_depth=4,
                min_samples_split=30,
                min_samples_leaf=15,
                max_features=0.4,
                bootstrap=True,
                random_state=self.seed
            )
            
        else:  # Wa(m)
            # 平衡的GradientBoosting
            return GradientBoostingRegressor(
                n_estimators=25,
                max_depth=3,
                learning_rate=0.04,
                subsample=0.6,
                min_samples_split=25,
                min_samples_leaf=12,
                random_state=self.seed
            )
    
    def fit_and_impute(self, data_original, mask, column_names, train_indices=None):
        """训练并填补 - 避免数据泄露"""
        simple_imputer = SimpleImputer(strategy='median')
        
        if train_indices is not None:
            train_data = data_original[train_indices]
            simple_imputer.fit(train_data)
        else:
            simple_imputer.fit(data_original)
        
        data_filled = simple_imputer.transform(data_original)
        imputed = data_original.copy()
        
        for col_idx, col_name in enumerate(column_names):
            if col_name not in self.target_features:
                continue
            
            if train_indices is not None:
                train_mask = np.zeros(len(data_original), dtype=bool)
                train_mask[train_indices] = True
                obs_mask = (mask[:, col_idx] == 1) & train_mask
            else:
                obs_mask = mask[:, col_idx] == 1
            
            if np.sum(obs_mask) < 50:
                continue
            
            # 训练模型
            X_train = np.delete(data_filled[obs_mask], col_idx, axis=1)
            y_train = data_original[obs_mask, col_idx]
            
            # 平衡的噪声正则化
            noise = np.random.normal(0, 0.008, X_train.shape) * np.std(X_train, axis=0)
            X_train_noisy = X_train + noise
            
            model = self._get_model_for_feature(col_name)
            model.fit(X_train_noisy, y_train)
            
            # 预测缺失值
            miss_mask = mask[:, col_idx] == 0
            if np.sum(miss_mask) > 0:
                X_miss = np.delete(data_filled[miss_mask], col_idx, axis=1)
                
                # 多次预测取平均
                predictions = []
                for _ in range(3):
                    noise = np.random.normal(0, 0.004, X_miss.shape) * np.std(X_miss, axis=0)
                    predictions.append(model.predict(X_miss + noise))
                
                imputed[miss_mask, col_idx] = np.mean(predictions, axis=0)
        
        return imputed

# 8. 平衡的最终集成模型
class FinalEnsemble:
    """平衡的最终集成模型"""
    def __init__(self, data_dim, seed=42):
        self.data_dim = data_dim
        self.seed = seed
        self.models = []
        self.weights = None
        self.scalers = None
        self.data_raw = None
        self.column_names = None
    
    def fit(self, data_normalized, mask, data_raw, scalers, column_names):
        """训练最终集成模型"""
        print("\n训练最终集成模型...")
        self.scalers = scalers
        self.data_raw = data_raw
        self.column_names = column_names
        
        # 创建增强特征
        data_enhanced, mask_enhanced = enhanced_feature_engineering(data_normalized, mask)
        print(f"增强特征维度: {data_enhanced.shape[1]}")
        
        # 1. KNN模型
        print("1. KNN模型...")
        knn = KNNImputer(n_neighbors=6, weights='distance')  # 平衡的邻居数
        knn.fit(data_normalized)
        self.models.append(('knn', knn))
        
        # 2. IterativeImputer
        print("2. IterativeImputer...")
        mice = IterativeImputer(
            estimator=BayesianRidge(),  # 使用贝叶斯岭回归
            max_iter=4,  # 平衡的迭代次数
            random_state=self.seed
        )
        mice.fit(data_normalized)
        self.models.append(('mice', mice))
        
        # 3. 优化的VAE
        print("3. VAE模型...")
        vae = OptimizedVAE(
            data_dim=self.data_dim,
            enhanced_dim=data_enhanced.shape[1],
            latent_dim=7,  # 平衡的潜在维度
            learning_rate=0.0006,  # 平衡的学习率
            seed=self.seed
        )
        
        mean_imputer = SimpleImputer(strategy='median')
        mean_imputer.fit(data_normalized)
        initial_imputed = mean_imputer.transform(data_normalized)
        data_filled = np.where(mask == 1, data_normalized, initial_imputed)
        
        data_filled_enhanced, _ = enhanced_feature_engineering(data_filled, mask)
        
        vae.fit(data_filled_enhanced, data_filled, mask, 
                validation_split=0.2, epochs=90, batch_size=96)
        self.models.append(('vae', vae))
        self.vae_enhanced_data = data_filled_enhanced
        
        # 4. 目标模型
        print("4. 目标特征模型（将在预测时按需训练）...")
        target_features = [col for col in column_names if col in ['H(m)', 'At(m²）', 'Wa(m)']]
        if not target_features:
            target_features = column_names[:min(3, len(column_names))]
        
        self.target_model = EnhancedTargetedModel(
            target_features=target_features,
            seed=self.seed
        )
        self.models.append(('targeted', self.target_model))
        
        print("集成模型训练完成")
        
        # 平衡的自适应权重
        self._compute_adaptive_weights()
    
    def _compute_adaptive_weights(self):
        """计算平衡的自适应权重"""
        print("\n计算自适应权重...")
        
        # 平衡的权重分配
        base_weights = {
            'knn': 0.27,      # 平衡的KNN权重
            'mice': 0.28,     # 平衡的MICE权重
            'vae': 0.27,      # 平衡的VAE权重
            'targeted': 0.18  # 平衡的目标模型权重
        }
        
        self.weights = np.array([base_weights[name] for name, _ in self.models])
        self.weights = self.weights / np.sum(self.weights)
        
        print(f"最终权重: {dict(zip([n for n, _ in self.models], self.weights))}")
    
    def impute(self, data_normalized, mask, column_names, train_indices=None):
        """最终集成预测"""
        predictions = []
        
        for (name, model), weight in zip(self.models, self.weights):
            if name == 'knn':
                pred = model.transform(data_normalized)
                pred_original = inverse_transform_optimized(pred, self.scalers, column_names)
                
            elif name == 'mice':
                pred = model.transform(data_normalized)
                pred_original = inverse_transform_optimized(pred, self.scalers, column_names)
                
            elif name == 'vae':
                mean_imputer = SimpleImputer(strategy='median')
                mean_imputer.fit(data_normalized)
                initial_imputed = mean_imputer.transform(data_normalized)
                data_filled = np.where(mask == 1, data_normalized, initial_imputed)
                data_filled_enhanced, _ = enhanced_feature_engineering(data_filled, mask)
                
                pred = model.impute(data_filled_enhanced, data_filled, mask)
                pred_original = inverse_transform_optimized(pred, self.scalers, column_names)
                
            elif name == 'targeted':
                pred_original = model.fit_and_impute(
                    self.data_raw, mask, column_names, train_indices
                )
            
            predictions.append(pred_original * weight)
        
        # 加权平均
        imputed = np.sum(predictions, axis=0)
        
        # 确保观测值不变
        imputed = self.data_raw * mask + imputed * (1 - mask)
        
        # 轻度后处理
        for col_idx in range(imputed.shape[1]):
            miss_mask = mask[:, col_idx] == 0
            if np.sum(miss_mask) > 2:
                # 使用轻度平滑
                from scipy.ndimage import gaussian_filter1d
                imputed[miss_mask, col_idx] = gaussian_filter1d(
                    imputed[miss_mask, col_idx], sigma=0.5
                )
        
        return imputed


# =================== 新增：综合评估函数修改 ===================
def comprehensive_evaluation(model, data_normalized, mask, column_names, data_raw, scalers):
    print("\n=== 最终评估（5折交叉验证 - 无数据泄露）===")
    results = []
    for col_idx, col_name in enumerate(column_names):
        obs_mask = mask[:, col_idx] == 1
        obs_indices = np.where(obs_mask)[0]
        if len(obs_indices) < 100:
            continue
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        all_true_original, all_pred_original, fold_metrics = [], [], []
        for fold, (train_idx, test_idx) in enumerate(kf.split(obs_indices)):
            train_indices = obs_indices[train_idx]
            test_indices = obs_indices[test_idx]
            test_mask = mask.copy()
            test_mask[test_indices, col_idx] = 0
            imputed_original = model.impute(data_normalized, test_mask, column_names, train_indices=train_indices)
            true_values = data_raw[test_indices, col_idx]
            pred_values = imputed_original[test_indices, col_idx]
            if len(true_values) > 1:
                fold_r2 = r2_score(true_values, pred_values)
                fold_mae = mean_absolute_error(true_values, pred_values)
                fold_metrics.append({'r2': fold_r2, 'mae': fold_mae})
            all_true_original.extend(true_values)
            all_pred_original.extend(pred_values)
        all_true_original = np.array(all_true_original)
        all_pred_original = np.array(all_pred_original)

        # ============== 【新增】输出真实值与预测值到CSV ==============
        try:
            save_df = pd.DataFrame({
                "Index": np.arange(len(all_true_original)),
                "True_Value": all_true_original,
                "Predicted_Value": all_pred_original
            })
            safe_name = col_name.replace("/", "_").replace("（", "_").replace("）", "_")
            save_df.to_csv(f"输出_{safe_name}_观测值_vs_预测值.csv", index=False, encoding='utf-8-sig')
        except Exception as e:
            print(f"⚠️ 保存 {col_name} 预测输出失败: {e}")
        # ============================================================

        mae = mean_absolute_error(all_true_original, all_pred_original)
        rmse = np.sqrt(mean_squared_error(all_true_original, all_pred_original))
        ss_res = np.sum((all_true_original - all_pred_original) ** 2)
        ss_tot = np.sum((all_true_original - np.mean(all_true_original)) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        r2_scores = [m['r2'] for m in fold_metrics]
        r2_std = np.std(r2_scores) if r2_scores else 0
        r2_mean = np.mean(r2_scores) if r2_scores else r2
        non_zero_mask = np.abs(all_true_original) > np.percentile(np.abs(all_true_original), 5)
        if np.sum(non_zero_mask) > 0:
            ape_values = np.abs((all_true_original[non_zero_mask] - all_pred_original[non_zero_mask]) / all_true_original[non_zero_mask])
            mape = np.mean(np.clip(ape_values, 0, np.percentile(ape_values, 95))) * 100
        else:
            mape = np.mean(np.abs(all_true_original - all_pred_original)) * 100
        correlation = np.corrcoef(all_true_original, all_pred_original)[0, 1] if np.std(all_true_original) > 0 and np.std(all_pred_original) > 0 else 0
        status = "优秀" if r2 >= 0.75 else "良好" if r2 >= 0.65 else "可接受" if r2 >= 0.55 else "需改进"
        results.append({
            '特征': col_name, 'CV平均R²': r2_mean, 'CV标准差': r2_std,
            '总体R²': r2, '相关系数': correlation, 'MAE': mae,
            'RMSE': rmse, 'MAPE(%)': mape, '状态': status
        })
        print(f"{col_name}: CV平均R²={r2_mean:.4f}±{r2_std:.4f}, 总体R²={r2:.4f}, MAPE={mape:.2f}%, 状态={status}")
    return pd.DataFrame(results)

# 10. 主函数
def main():
    """主函数"""
    print("=" * 80)
    print("高性能VAE缺失值填补模型 - 平衡优化版本")
    print("主要改进：")
    print("1. 平衡的正则化强度")
    print("2. 优化的模型复杂度")
    print("3. 混合损失函数")
    print("4. 平衡的集成权重")
    print("5. 改进的评估指标")
    print("=" * 80)
    
    try:
        set_all_seeds(SEED)
        
        # 1. 加载数据
        print("\n1. 加载数据...")
        data_raw, original_data = load_and_preprocess_data()
        
        # 2. 自适应异常值处理
        print("\n2. 自适应异常值处理...")
        data_clean, transform_info = adaptive_outlier_handling(data_raw)
        
        # 3. 优化的标准化
        print("\n3. 优化的混合标准化...")
        data_normalized, scalers = optimized_normalization(data_clean)
        
        # 创建掩码
        mask = (~data_normalized.isnull()).astype(np.float32).values
        data_array = data_normalized.fillna(0.5).values.astype(np.float32)
        data_raw_array = data_clean.fillna(0).values.astype(np.float32)
        
        print(f"\n数据形状: {data_array.shape}")
        print(f"缺失率: {(1 - mask.mean())*100:.2f}%")
        
        # 显示标准化策略
        print("\n标准化策略：")
        for col_name, scaler_info in scalers.items():
            print(f"  {col_name}: {scaler_info['type']}")
        
        # 4. 训练最终集成模型
        print("\n4. 训练最终集成模型...")
        model = FinalEnsemble(data_dim=data_array.shape[1], seed=SEED)
        model.fit(data_array, mask, data_raw_array, scalers, data_normalized.columns)
        
        # 5. 评估（无数据泄露）
        print("\n5. 模型评估（无数据泄露）...")
        eval_results = comprehensive_evaluation(
            model, data_array, mask, data_normalized.columns,
            data_raw_array, scalers
        )
        
        # 6. 生成填补数据（使用所有数据）
        print("\n6. 生成最终填补数据...")
        final_imputed_original = model.impute(data_array, mask, data_normalized.columns)
        
        # 7. 输出结果
        print("\n" + "=" * 80)
        print("最终模型性能（平衡优化版本）")
        print("=" * 80)
        
        for _, row in eval_results.iterrows():
            print(f"\n{row['特征']}:")
            print(f"  CV平均R² = {row['CV平均R²']:.4f} ± {row['CV标准差']:.4f}")
            print(f"  总体R² = {row['总体R²']:.4f}")
            print(f"  MAE = {row['MAE']:.4f}")
            print(f"  RMSE = {row['RMSE']:.4f}")
            print(f"  MAPE = {row['MAPE(%)']:.2f}%")
            print(f"  状态: {row['状态']}")
        
        # 汇总统计
        print("\n" + "-" * 40)
        print("汇总统计")
        print("-" * 40)
        
        avg_cv_r2 = eval_results['CV平均R²'].mean()
        avg_total_r2 = eval_results['总体R²'].mean()
        avg_mape = eval_results['MAPE(%)'].mean()
        avg_cv_std = eval_results['CV标准差'].mean()
        
        print(f"平均CV R²: {avg_cv_r2:.4f} ± {avg_cv_std:.4f}")
        print(f"平均总体R²: {avg_total_r2:.4f}")
        print(f"平均MAPE: {avg_mape:.2f}%")
        
        # 稳定性分析
        print(f"\n稳定性分析:")
        for _, row in eval_results.iterrows():
            print(f"  {row['特征']}: CV标准差={row['CV标准差']:.4f}")
        
        # 保存结果
        final_df_original = pd.DataFrame(final_imputed_original, columns=data_normalized.columns)
        final_df_original.to_csv("填补结果_Earthquake_平衡版.csv", index=False)
        eval_results.to_csv("评估结果_Earthquake_平衡版.csv", index=False)
        
        print("\n文件已保存：")
        print("- 填补结果_Earthquake_平衡版.csv")
        print("- 评估结果_Earthquake_平衡版.csv")
        
        print("\n" + "=" * 50)
        print("最终诊断")
        print("=" * 50)
        
        # 综合诊断
        if avg_cv_std < 0.035:
            print("✅ 模型稳定性优秀")
        elif avg_cv_std < 0.06:
            print("✅ 模型稳定性良好")
        else:
            print("⚠️ 模型稳定性需要改进")
        
        if avg_total_r2 >= 0.75:
            print("✅ 预测性能优秀")
        elif avg_total_r2 >= 0.65:
            print("✅ 预测性能良好")
        elif avg_total_r2 >= 0.55:
            print("✅ 预测性能可接受")
        else:
            print("⚠️ 预测性能需要提高")
        
        if avg_mape < 20:
            print("✅ MAPE表现优秀")
        elif avg_mape < 25:
            print("✅ MAPE表现良好")
        else:
            print("⚠️ MAPE需要改进")
        
        # 拟合状态判断
        if avg_total_r2 >= 0.75 and avg_total_r2 <= 0.85:
            print("\n✅ 模型拟合平衡，无明显过拟合或欠拟合")
        elif avg_total_r2 > 0.85:
            print("\n⚠️ 可能存在轻微过拟合")
        else:
            print("\n⚠️ 可能存在轻微欠拟合")
        
        print("✅ 评估结果可靠，适合用于研究")
        
        return final_imputed_original, eval_results
        
    except Exception as e:
        print(f"\n错误: {e}")
        import traceback
        traceback.print_exc()
        return None, None

if __name__ == "__main__":
    print("\n程序开始运行...")
    print("优化目标：")
    print("1. 平衡过拟合与欠拟合")
    print("2. 提高模型泛化能力")
    print("3. 保持良好的预测精度")
    print("4. 确保结果稳定可靠")
    
    start_time = time.time()
    result, evaluation = main()
    
    if result is not None:
        print(f"\n✅ 程序运行完成！")
        print(f"总用时: {(time.time() - start_time)/60:.2f} 分钟")
        
        # 性能总结
        if evaluation is not None:
            print("\n" + "=" * 50)
            print("性能总结")
            print("=" * 50)
            
            avg_total_r2 = evaluation['总体R²'].mean()
            avg_cv_r2 = evaluation['CV平均R²'].mean()
            avg_mape = evaluation['MAPE(%)'].mean()
            
            print(f"最终平均总体R²: {avg_total_r2:.4f}")
            print(f"最终平均CV R²: {avg_cv_r2:.4f}")
            print(f"最终平均MAPE: {avg_mape:.2f}%")
            print(f"R²范围: [{evaluation['总体R²'].min():.4f}, {evaluation['总体R²'].max():.4f}]")
            
            print("\n🎉 模型优化完成！")
            print("已达到最佳平衡状态。")
            
            # 改进建议
            if avg_total_r2 < 0.75:
                print("\n建议：如需进一步提升性能，可考虑增加数据量或特征工程")
            else:
                print("\n模型性能良好，适合用于实际研究")
            
            print("\n感谢使用！")
    else:
        print("\n❌ 运行失败，请检查错误信息")
    
    gc.collect()

# ===========================================================
# 启动入口
# ===========================================================
if __name__ == "__main__":
    print("\n程序开始运行...")
    start_time = time.time()
    result, evaluation = main()
    if result is not None:
        print("\n✅ 所有特征的观测值与预测值已保存为 CSV 文件。")


TensorFlow版本: 2.21.0
未发现GPU，使用CPU运行


程序开始运行...
优化目标：
1. 平衡过拟合与欠拟合
2. 提高模型泛化能力
3. 保持良好的预测精度
4. 确保结果稳定可靠
高性能VAE缺失值填补模型 - 平衡优化版本
主要改进：
1. 平衡的正则化强度
2. 优化的模型复杂度
3. 混合损失函数
4. 平衡的集成权重
5. 改进的评估指标

1. 加载数据...

错误: [Errno 2] No such file or directory: './Earthquake.csv'

❌ 运行失败，请检查错误信息

程序开始运行...
高性能VAE缺失值填补模型 - 平衡优化版本
主要改进：
1. 平衡的正则化强度
2. 优化的模型复杂度
3. 混合损失函数
4. 平衡的集成权重
5. 改进的评估指标

1. 加载数据...

错误: [Errno 2] No such file or directory: './Earthquake.csv'


Traceback (most recent call last):
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23856\202605219.py", line 854, in main
    data_raw, original_data = load_and_preprocess_data()
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23856\202605219.py", line 64, in load_and_preprocess_data
    data = pd.read_csv("./Earthquake.csv", low_memory=True)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 873, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 300, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line

In [2]:
import pandas as pd

def add_L_column_to_results():
    """
    将L(m)列添加到已生成的填补结果文件中
    """
    # 1. 读取原始数据以获取L(m)列
    original_data = pd.read_csv("./Earthquake.csv", low_memory=True)
    
    # 2. 检查L(m)列是否存在
    if 'L(m)' not in original_data.columns:
        print("原始数据中未找到L(m)列")
        return
    
    # 3. 提取L(m)列
    L_column = original_data['L(m)'].copy()
    print(f"成功提取L(m)列，共{len(L_column)}行")
    
    # 4. 读取填补结果
    try:
        imputed_results = pd.read_csv("填补结果_Earthquake_平衡版.csv")
        print(f"填补结果形状: {imputed_results.shape}")
    except FileNotFoundError:
        print("未找到填补结果文件，请先运行主程序生成结果")
        return
    
    # 5. 检查行数是否匹配
    if len(imputed_results) != len(L_column):
        print(f"警告：行数不匹配！填补结果有{len(imputed_results)}行，L(m)列有{len(L_column)}行")
        # 如果行数不同，需要对齐（取最小行数）
        min_rows = min(len(imputed_results), len(L_column))
        imputed_results = imputed_results.iloc[:min_rows]
        L_column = L_column.iloc[:min_rows]
    
    # 6. 添加L(m)列到结果中
    imputed_results['L(m)'] = L_column.values
    
    # 7. 保存更新后的结果
    output_filename = "填补结果_Earthquake_包含L列_平衡版.csv"
    imputed_results.to_csv(output_filename, index=False)
    
    print(f"\n成功添加L(m)列到结果中！")
    print(f"新文件已保存为: {output_filename}")
    print(f"最终数据形状: {imputed_results.shape}")
    print(f"列名: {list(imputed_results.columns)}")
    
    # 8. 显示L(m)列的基本统计信息
    print("\nL(m)列统计信息:")
    print(f"  数据类型: {L_column.dtype}")
    print(f"  唯一值数量: {L_column.nunique()}")
    print(f"  缺失值数量: {L_column.isna().sum()}")
    if pd.api.types.is_numeric_dtype(L_column):
        print(f"  最小值: {L_column.min()}")
        print(f"  最大值: {L_column.max()}")
        print(f"  平均值: {L_column.mean():.2f}")
    else:
        print(f"  前5个值: {L_column.head().tolist()}")
    
    return imputed_results

# 运行函数
if __name__ == "__main__":
    result_with_L = add_L_column_to_results()

成功提取L(m)列，共4330行
填补结果形状: (4330, 3)

成功添加L(m)列到结果中！
新文件已保存为: 填补结果_Earthquake_包含L列_平衡版.csv
最终数据形状: (4330, 4)
列名: ['H(m)', 'At(m²）', 'Wa(m)', 'L(m)']

L(m)列统计信息:
  数据类型: float64
  唯一值数量: 1141
  缺失值数量: 0
  最小值: 0.26
  最大值: 44000.0
  平均值: 279.66
